# Phase 1 - Step 1: Data Understanding

This notebook dynamically discovers, inspects, and analyzes all raw datasets in `data/raw` for the Enterprise HR AI platform.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(list(raw_dir.glob("*.csv")))
print(f"Discovered {len(csv_files)} raw datasets:")
for f in csv_files:
    print(f" - {f.name}")


Discovered 5 raw datasets:
 - employee_attrition.csv
 - essential_skills.csv
 - hr_performance_engagement.csv
 - occupation_data.csv
 - software_skills.csv


In [2]:
summary_records = []

for f in csv_files:
    print("=" * 70)
    print(f"ANALYZING: {f.name}")
    df = pd.read_csv(f)
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Missing and duplicate counts
    missing_count = int(df.isnull().sum().sum())
    dup_count = int(df.duplicated().sum())
    
    # ID & Join key candidates
    id_candidates = [c for c in df.columns if any(k in c.lower() for k in ["id", "code", "title", "role"])]
    
    # Likely primary key
    likely_pk = "None"
    for c in df.columns:
        if df[c].nunique() == len(df) and df[c].isnull().sum() == 0:
            likely_pk = c
            break
            
    # Purpose inference
    purpose_map = {
        "employee_attrition.csv": "Employee attrition records, demographics, and risk target",
        "essential_skills.csv": "O*NET essential skills baseline for occupations",
        "hr_performance_engagement.csv": "Employee performance, attendance, ratings, and engagement metrics",
        "occupation_data.csv": "O*NET standard occupation codes, job titles, and descriptions",
        "software_skills.csv": "Software, tools, and technical skill requirements per occupation"
    }
    
    # Target detection for attrition
    target_info = "N/A"
    if "attrition" in f.name.lower():
        target_cols = [c for c in df.columns if "attrition" in c.lower() or "target" in c.lower() or "left" in c.lower()]
        if target_cols:
            t_col = target_cols[0]
            dist = df[t_col].value_counts().to_dict()
            target_info = f"Target: {t_col} -> {dist}"
            print(f"Target Distribution for {t_col}:\n{df[t_col].value_counts(dropna=False)}")
            
    summary_records.append({
        "Dataset": f.name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Likely Primary Key": likely_pk,
        "Missing Values": missing_count,
        "Duplicates": dup_count,
        "Potential Join Key": ", ".join(id_candidates[:3]),
        "Purpose": purpose_map.get(f.name, "Raw HR Dataset")
    })
    
    print("\nColumns & Dtypes:")
    print(df.dtypes)
    print("\nMissing Values per column:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print("\nHead (3 rows):")
    print(df.head(3))


ANALYZING: employee_attrition.csv
Shape: 500 rows, 24 columns
Target Distribution for AttritionRisk:
AttritionRisk
No     445
Yes     55
Name: count, dtype: int64

Columns & Dtypes:
EmployeeID                 int64
Name                         str
Gender                       str
Age                        int64
Department                   str
JobRole                      str
EducationLevel             int64
JoiningDate                  str
CountryCode                int64
Country                      str
PhoneNumber                int64
MonthlySalary              int64
OvertimeHoursPerMonth      int64
LeavesTaken                int64
LastLeaveDate                str
LeaveDayName                 str
ProjectsHandled            int64
TrainingHours              int64
CustomerSatisfaction     float64
LastPromotionYear          int64
YearsAtCompany             int64
WorkLifeBalanceScore     float64
PerformanceRating          int64
AttritionRisk                str
dtype: object

Missing Val

Shape: 31821 rows, 7 columns

Columns & Dtypes:
O*NET-SOC Code       str
Title                str
Workplace Example    str
Element ID           str
Element Name         str
Hot Technology       str
In Demand            str
dtype: object

Missing Values per column:
Series([], dtype: int64)

Head (3 rows):
  O*NET-SOC Code             Title Workplace Example Element ID  \
0     11-1011.00  Chief Executives     Adobe Acrobat    2.E.5.b   
1     11-1011.00  Chief Executives   AdSense Tracker    2.E.6.f   
2     11-1011.00  Chief Executives    Atlassian JIRA    2.E.5.a   

                                  Element Name Hot Technology In Demand  
0                 Document management software              Y         N  
1  Data base user interface and query software              N         N  
2                    Content workflow software              Y         N  


In [3]:
summary_df = pd.DataFrame(summary_records)
summary_csv_path = processed_dir / "data_understanding_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"Saved understanding summary to {summary_csv_path}")
print(summary_df.to_string())


Saved understanding summary to data\processed\data_understanding_summary.csv
                         Dataset   Rows  Columns Likely Primary Key  Missing Values  Duplicates                 Potential Join Key                                                            Purpose
0         employee_attrition.csv    500       24         EmployeeID             319           0   EmployeeID, JobRole, CountryCode          Employee attrition records, demographics, and risk target
1           essential_skills.csv  18200       15               None            9100           0  O*NET-SOC Code, Title, Element ID                    O*NET essential skills baseline for occupations
2  hr_performance_engagement.csv   5000       13        Employee ID               0           0              Employee ID, Job Role  Employee performance, attendance, ratings, and engagement metrics
3            occupation_data.csv   1016        3     O*NET-SOC Code               0           0              O*NET-SOC Code, Title 